# Text Extraction as per columns

In [1]:
!pip install pdfplumber

### Step 1: Column-wise PDF Text Extraction

**Purpose:**  
Extract text from a multi-column NCERT PDF while preserving correct reading order.

**What this code does:**  
- Splits each page into left and right columns  
- Extracts text separately  
- Combines them in correct sequence  

**Why this is important:**  
Default extraction mixes columns → breaks sentences → poor chunking & retrieval.

In [2]:
import pdfplumber

pdf_path = "iesc107_removed (1).pdf"

final_text = ""

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        width = page.width
        height = page.height

        # Define column regions
        left_bbox = (0, 0, width/2, height)
        right_bbox = (width/2, 0, width, height)

        # Crop columns
        left_column = page.crop(left_bbox)
        right_column = page.crop(right_bbox)

        # Extract text in correct reading order
        left_text = left_column.extract_text() or ""
        right_text = right_column.extract_text() or ""

        final_text += left_text + "\n"
        final_text += right_text + "\n"

# Save output
with open("output.txt", "w", encoding="utf-8") as f:
    f.write(final_text)

print(final_text)

7
C
hapter
In everyday life, we see some objects at rest
and others in motion. Birds fly, fish swim,
blood flows through veins and arteries, and
cars move. Atoms, molecules, planets, stars
and galaxies are all in motion. We often
perceive an object to be in motion when its
position changes with time. However, there
are situations where the motion is inferred
through indirect evidences. For example, we
infer the motion of air by observing the
movement of dust and the movement of leaves
and branches of trees. What causes the
phenomena of sunrise, sunset and changing
of seasons? Is it due to the motion of the
earth? If it is true, why don’t we directly
perceive the motion of the earth?
An object may appear to be moving for
one person and stationary for some other. For
the passengers in a moving bus, the roadside
trees appear to be moving backwards. A
person standing on the road–side perceives
the bus alongwith the passengers as moving.
However, a passenger inside the bus sees his
fellow p

### 🔹 Step 2: Fix Broken Headings

**Purpose:**  
Correct headings split across multiple lines due to PDF extraction issues.

**What this code does:**  
- Merges split words like "U - NIFORM"  
- Fixes section headings like "7.1.2"  
- Ensures headings are readable and complete  

**Why this is important:**  
Incorrect headings break semantic structure and chunk grouping.

In [3]:
import re

def fix_split_headings_final(text):
    lines = text.split("\n")
    fixed_lines = []
    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()

        # -----------------------------
        # Detect heading start: 7.x or 7.x.x
        # -----------------------------
        if re.match(r'^\d+\.\d+(\.\d+)?', line):
            combined = line

            j = i + 1

            # Merge next 1–2 lines (headings are short)
            while j < n and len(combined.split()) < 8:
                next_line = lines[j].strip()

                # stop if next is new section or empty
                if re.match(r'^\d+\.\d+', next_line):
                    break
                if next_line == "":
                    break

                combined += " " + next_line
                j += 1

            # Clean broken patterns inside heading
            combined = re.sub(r'([A-Z])\s*[-–]\s*([A-Z])', r'\1\2', combined)
            combined = re.sub(r'\b([A-Z])\s+([A-Z]+)', r'\1\2', combined)

            fixed_lines.append(combined)
            i = j
            continue

        # -----------------------------
        # Fix standalone broken words
        # -----------------------------
        if i + 1 < n:
            next_line = lines[i + 1].strip()

            if (
                len(line) == 1 and line.isalpha()
                and next_line.isalpha()
            ):
                combined = line + next_line
                fixed_lines.append(combined)
                i += 2
                continue

        fixed_lines.append(line)
        i += 1

    return "\n".join(fixed_lines)

### Step 3: Text Cleaning Pipeline

**Purpose:**  
Remove noise and fix structural issues in extracted text.

**What this code does:**  
- Fixes repeated characters (e.g., MMMOTION → MOTION)  
- Joins spaced words (M O T I O N → MOTION)  
- Fixes broken headings  
- Removes unnecessary repeated headers  

**Why this is important:**  
Clean text ensures:
- Better tokenization  
- Better chunk boundaries  
- Improved retrieval accuracy  

In [5]:
import pdfplumber
import re

def fix_repeated_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)

def fix_spaced_words(text):
    return re.sub(r'\b(?:[A-Za-z]\s){2,}[A-Za-z]\b',
                  lambda m: m.group(0).replace(" ", ""),
                  text)

def fix_heading_lines(text):
    lines = text.split("\n")
    fixed_lines = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        # Case 1: Chapter number + broken "Chapter"
        if (
            i + 2 < len(lines)
            and lines[i].strip().isdigit()
            and lines[i+1].strip().lower() in ["c", "ch"]
            and re.match(r"hapter", lines[i+2].strip(), re.IGNORECASE)
        ):
            chapter_num = lines[i].strip()
            fixed_lines.append(f"Chapter {chapter_num}")
            i += 3
            continue

        # Case 2: Broken ALL CAPS word like M + OTION
        if (
            i + 1 < len(lines)
            and len(lines[i].strip()) == 1
            and lines[i].strip().isalpha()
            and lines[i+1].strip().isalpha()
        ):
            combined = lines[i].strip() + lines[i+1].strip()
            fixed_lines.append(combined)
            i += 2
            continue

        # Default: keep line
        fixed_lines.append(line)
        i += 1

    return "\n".join(fixed_lines)

def extract_clean_text(pdf_path):
    all_text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=2, y_tolerance=2)

            if text:
                text = fix_repeated_chars(text)
                text = fix_spaced_words(text)
                text = fix_heading_lines(text) # Apply the heading fix
                text = fix_chapter_motion_position(text)
                all_text += text + "\n"

    return all_text

In [6]:
def fix_chapter_motion_position(text):
    lines = text.split("\n")

    cleaned_lines = []
    motion_line = None

    # Step 1: normalize MOTION (already repeated chars fixed earlier)
    for line in lines:
        stripped = line.strip()

        # detect MOTION (after repeated char fix it becomes "MOTION")
        if stripped.upper() == "MOTION":
            motion_line = "MOTION"
            continue

        # remove junk like MMMMM, etc. (already mostly handled)
        if re.match(r'^[A-Z]{2,}$', stripped) and stripped != "MOTION":
            continue

        cleaned_lines.append(line)

    # Step 2: insert MOTION after Chapter line
    final_lines = []
    inserted = False

    for line in cleaned_lines:
        final_lines.append(line)

        if not inserted and re.match(r'^Chapter\s*\d+', line, re.IGNORECASE):
            if motion_line:
                final_lines.append(motion_line)
                inserted = True

    return "\n".join(final_lines)

In [7]:
# Apply cleaning on column-extracted text
text = final_text

text = fix_repeated_chars(text)
text = fix_spaced_words(text)
text = fix_heading_lines(text)
text = fix_split_headings_final(text)
text = fix_chapter_motion_position(text)

# Save cleaned output
#with open("cleaned_output.txt", "w", encoding="utf-8") as f:
    #f.write(text)

print(text)

Chapter 7
MOTION
In everyday life, we see some objects at rest
and others in motion. Birds fly, fish swim,
blood flows through veins and arteries, and
cars move. Atoms, molecules, planets, stars
and galaxies are all in motion. We often
perceive an object to be in motion when its
position changes with time. However, there
are situations where the motion is inferred
through indirect evidences. For example, we
infer the motion of air by observing the
movement of dust and the movement of leaves
and branches of trees. What causes the
phenomena of sunrise, sunset and changing
of seasons? Is it due to the motion of the
earth? If it is true, why don’t we directly
perceive the motion of the earth?
An object may appear to be moving for
one person and stationary for some other. For
the passengers in a moving bus, the roadside
trees appear to be moving backwards. A
person standing on the road–side perceives
the bus alongwith the passengers as moving.
However, a passenger inside the bus sees his
fe

###  Step 4: Remove Figure References and Non-Contextual Elements

**Purpose:**  
Remove figure numbers and their descriptions from the extracted text, as they are not useful for textual understanding and can introduce noise in downstream tasks.

**What this code does:**  
- Removes figure captions such as `Fig. 7.1`, `Figure 7.2`, etc.  
- Removes inline references like `(Fig. 7.3)` within sentences  
- Cleans leftover incomplete phrases after figure removal  
- Preserves meaningful content such as explanations, definitions, and concepts  

**Why this is important:**  
Figure references:
- Do not contribute to semantic understanding in a text-only system  
- Can confuse retrieval models  
- Increase unnecessary tokens  

Removing them ensures:
- Cleaner input for tokenization  
- Better chunk quality  
- Improved retrieval accuracy  
  

In [8]:
import re

def preprocess_text(text):

    # -----------------------------
    # 1. Remove bullets & symbols
    # -----------------------------
    text = re.sub(r'[•●▪■▪◦◆►]', '', text)
    text = re.sub(r'[_]', '', text)

    # -----------------------------
    # 2. Remove Reprint / year lines
    # -----------------------------
    text = re.sub(r'Reprint\s*\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d{4}-\d{2}\b', '', text)   # 2025-26
    text = re.sub(r'\b\d{4}\b', '', text)         # standalone year
    
    # -----------------------------
    # 3. Remove figure captions (FULL LINES)
    # -----------------------------
    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        # Remove lines like:
        # Fig. 7.1: ...
        # Figure 7.2 ...
        if re.match(r'^(Fig\.?|Figure)\s*\d+(\.\d+)?', stripped, re.IGNORECASE):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    
    # -----------------------------
    # 4. Remove inline figure refs
    # -----------------------------
    text = re.sub(r'\(?\bFig\.?\s*\d+(\.\d+)?[a-zA-Z]?\)?', '', text, flags=re.IGNORECASE)

    # -----------------------------
    # 5. Clean leftover broken phrases
    # -----------------------------
    text = re.sub(r'\b(in|shown in|given in)\s*\.', '', text)

    # -----------------------------
    # 6. Clean spacing (KEEP paragraphs)
    # -----------------------------
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    text = "\n".join(lines)

    return text

In [9]:
clean_text = preprocess_text(text)

# Save output
#with open("cleaned_text.txt", "w", encoding="utf-8") as f:
    #f.write(clean_text)

print(clean_text)

Chapter 7
MOTION
In everyday life, we see some objects at rest
and others in motion. Birds fly, fish swim,
blood flows through veins and arteries, and
cars move. Atoms, molecules, planets, stars
and galaxies are all in motion. We often
perceive an object to be in motion when its
position changes with time. However, there
are situations where the motion is inferred
through indirect evidences. For example, we
infer the motion of air by observing the
movement of dust and the movement of leaves
and branches of trees. What causes the
phenomena of sunrise, sunset and changing
of seasons? Is it due to the motion of the
earth? If it is true, why don’t we directly
perceive the motion of the earth?
An object may appear to be moving for
one person and stationary for some other. For
the passengers in a moving bus, the roadside
trees appear to be moving backwards. A
person standing on the road–side perceives
the bus alongwith the passengers as moving.
However, a passenger inside the bus sees his
fe

###  Output Analysis

**Observations:**
- All figure-related noise is removed  
- Paragraph structure remains intact  
- Conceptual content is preserved  

**Impact:**
- Text becomes more readable and coherent  
- No irrelevant visual references remain  
- Improved suitability for LLM-based processing

###  Data Structuring Strategy

Instead of deleting non-essential content (Activities, Examples, Questions),  
these sections are **separated and preserved** in individual files.

This approach:
- Prevents data loss  
- Enables future reuse  
- Improves modularity of the dataset  

### Step 5: Remove Non-Contextual Sections

**Purpose:**  
Remove sections not useful for question answering.

**What this code does:**  
- Extracts "Activity" sections  
- Extracts "Think and Act" sections  
- Saves them into separate files  

**Why this is important:**  
These sections:
- Do not contain factual knowledge  
- Add noise to retrieval system  

In [10]:
import re

def extract_sections(text):
    lines = text.split("\n")

    main_text = []
    activities = []
    think_act = []

    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()

        # -----------------------------
        # ACTIVITY BLOCK
        # -----------------------------
        if re.match(r'^Activity\s*\d+(\.\d+)?', line, re.IGNORECASE):
            temp = [line]
            i += 1

            while i < n:
                next_line = lines[i].strip()

                # STOP conditions (very important)
                if (
                    re.match(r'^(Activity|Think and Act|Questions|Example|\d+\.\d+)', next_line, re.IGNORECASE)
                ):
                    break

                temp.append(next_line)
                i += 1

            activities.append("\n".join(temp))
            continue

        # -----------------------------
        # THINK AND ACT BLOCK
        # -----------------------------
        if re.match(r'^Think and Act', line, re.IGNORECASE):
            temp = [line]
            i += 1

            while i < n:
                next_line = lines[i].strip()

                if re.match(r'^(Activity|Questions|Example|\d+\.\d+)', next_line, re.IGNORECASE):
                    break

                temp.append(next_line)
                i += 1

            think_act.append("\n".join(temp))
            continue

        # -----------------------------
        # NORMAL CONTENT
        # -----------------------------
        main_text.append(line)
        i += 1

    return "\n".join(main_text), activities, think_act

In [14]:
# Step 1: Load cleaned text
with open("cleaned_text.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Step 2: Extract sections
main_text, activities, think_act = extract_sections(text)

# Step 3: Save activities
with open("activities.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(activities))

# Step 4: Save Think & Act
with open("thinkandact.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(think_act))

# Step 5: Save cleaned main text
with open("final.txt", "w", encoding="utf-8") as f:
    f.write(main_text)

print(main_text)

Chapter 7
MOTION
In everyday life, we see some objects at rest
and others in motion. Birds fly, fish swim,
blood flows through veins and arteries, and
cars move. Atoms, molecules, planets, stars
and galaxies are all in motion. We often
perceive an object to be in motion when its
position changes with time. However, there
are situations where the motion is inferred
through indirect evidences. For example, we
infer the motion of air by observing the
movement of dust and the movement of leaves
and branches of trees. What causes the
phenomena of sunrise, sunset and changing
of seasons? Is it due to the motion of the
earth? If it is true, why don’t we directly
perceive the motion of the earth?
An object may appear to be moving for
one person and stationary for some other. For
the passengers in a moving bus, the roadside
trees appear to be moving backwards. A
person standing on the road–side perceives
the bus alongwith the passengers as moving.
However, a passenger inside the bus sees his
fe

###  Step 6: Extract Questions Section

**Purpose:**  
Separate evaluation-based content from core learning material.

**What this code does:**  
- Detects "Questions" blocks  
- Removes them from main text  
- Saves into questions.txt  

**Why this is important:**  
Useful for:
- Future evaluation datasets  
- Testing QA system

In [16]:
import re

with open('final.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

questions_text = []
main_text = []

in_questions = False

# Regex perfectly targets section headers (e.g., "7.1 Describing", "7.1.1 MOTION"), Examples, and Activities.
# It requires an uppercase letter after the numbers, which prevents "0.1 m s-2" from triggering it!
end_pattern = re.compile(r'^(\d+\.\d+(\.\d+)?\s+[A-Z]|Example\s+\d+\.\d+|Activity\s+\d+\.\d+)')

for line in lines:
    stripped_line = line.strip()
    
    # Check if we are starting a questions block
    if stripped_line == 'Questions' or stripped_line == 'Q uestions':
        in_questions = True
        questions_text.append(line)
        continue
        
    if in_questions:
        # Check if we hit a new section header
        if end_pattern.match(stripped_line):
            in_questions = False
            main_text.append(line)
        # Check if it's a stray page number (just digits)
        elif re.match(r'^\d+$', stripped_line):
            main_text.append(line) # Put page numbers back in the main text
        else:
            questions_text.append(line)
    else:
        main_text.append(line)

with open('Questions.txt', 'w', encoding='utf-8') as f:
    f.writelines(questions_text)

with open('final_cleaned_text.txt', 'w', encoding='utf-8') as f:
    f.writelines(main_text)

print(f"Extraction complete!")
print(f"Extracted {len(questions_text)} lines of Questions and saved them to Questions.txt")
print(f"Main text without questions saved to final_cleaned_text.txt")


Extraction complete!
Extracted 93 lines of Questions and saved them to Questions.txt
Main text without questions saved to final_cleaned_text.txt


### Step 7: Extract and Remove Examples Section

**Purpose:**  
Separate example-based content from the main text to focus on core conceptual information while preserving examples for future use.

**What this code does:**  
- Detects "Example" sections within the text  
- Extracts complete example blocks  
- Removes them from the main text  
- Saves extracted examples into a separate file (`examples.txt`)  

**Why this is important:**  
Examples:
- Often contain step-by-step solutions or illustrations  
- May introduce unnecessary verbosity for retrieval tasks  
- Can dilute the importance of core concepts  

By separating them:
- Main text becomes more concise and focused  
- Examples can be reused later for:
  - Training QA systems  
  - Evaluation datasets  
  - Practice generation 

In [17]:
import re

with open('final_cleaned_text.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

examples_text = []
main_text = []

in_example = False

# Matches starts of examples like "Example 7.1", "Example 7.2", or "7.4 Example"
start_pattern = re.compile(r'^(Example\s*\d+\.\d+|\d+\.\d+\s+Example)')

# Matches new sections (e.g., "7.1 Describing", "7.1.1 MOTION") or "Activity 7.1"
end_pattern = re.compile(r'^(\d+\.\d+(\.\d+)?\s+[A-Z]|Activity\s+\d+\.\d+)')

for line in lines:
    stripped_line = line.strip()
    
    # Check if we hit an Example start
    if start_pattern.match(stripped_line):
        in_example = True
        examples_text.append(line)
        continue
        
    if in_example:
        # Check if we hit a new section header or activity (which means the example ended)
        if end_pattern.match(stripped_line):
            in_example = False
            main_text.append(line)
        # Put stray page numbers back into the main text
        elif re.match(r'^\d+$', stripped_line):
            main_text.append(line)
        else:
            examples_text.append(line)
    else:
        main_text.append(line)

with open('Examples.txt', 'w', encoding='utf-8') as f:
    f.writelines(examples_text)

# Overwrite final_cleaned_text.txt so it now has NO questions AND NO examples.
with open('final_cleaned_text.txt', 'w', encoding='utf-8') as f:
    f.writelines(main_text)

print(f"Extraction complete!")
print(f"Extracted {len(examples_text)} lines of Examples and saved them to Examples.txt")
print(f"Main text without examples saved to final_cleaned_text.txt")


Extraction complete!
Extracted 176 lines of Examples and saved them to Examples.txt
Main text without examples saved to final_cleaned_text.txt


#### Removing "What you have learnt" from Activities.txt file

In [ ]:
output_lines = []
buffer = []

with open("activities.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    # Check for the 3-line pattern
    if (
        i + 2 < len(lines)
        and lines[i].strip() == "What"
        and lines[i + 1].strip() == "you have"
        and lines[i + 2].strip() == "learnt"
    ):
        break  # Stop processing from here onward

    output_lines.append(lines[i])
    i += 1

# Write back (or to a new file)
with open("activities_cleaned.txt", "w", encoding="utf-8") as f:
    f.writelines(output_lines)

#  Final Summary

##  Work Completed
- Extracted text from multi-column PDF using column-wise parsing  
- Cleaned text by removing noise, special characters, and formatting issues  
- Fixed broken words and headings caused by PDF extraction  
- Preserved metadata: **Chapter 7 MOTION** at the beginning  
- Removed non-contextual content:
  - Figures and captions  
  - Activities  
  - Think and Act  
  - Questions  
  - "What you have learnt"  
- Extracted and stored removed sections separately:
  - activities.txt  
  - thinkandact.txt  
  - questions.txt  
  - examples.txt  

## Remaining Work
- Still pending to work on "What have you learnt"
- Extract Chapter End Questions
- Separate formulaes from extracted text file 

